# Gain-modulation perturbations

This notebook evaluates three gain operators across all five frozen fixation-gated working-memory checkpoints. Every gain is applied at every timestep of the complete trial. Performance is measured on four deterministic 64-trial batches per checkpoint and expressed as the paired change from that checkpoint's own baseline.

Each gain section produces one figure: paired checkpoint-level performance effects plus baseline and perturbed trajectories in a joint PCA space for a representative checkpoint. PCA is not pooled across checkpoints because independently trained hidden-unit bases are not aligned.

## 1. Imports and paths

In [ ]:
from copy import deepcopy
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch


def find_repo(start: Path) -> Path:
    """Find the repository root from either the repo or notebooks directory."""
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'wm_rnn').is_dir() and (candidate / 'configs').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the working-memory-rnn repository root.')


REPO = find_repo(Path.cwd().resolve())
SRC = REPO / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from wm_rnn.config import load_config
from wm_rnn.device import select_device
from wm_rnn.training_utils import (
    batch_to_tensors,
    fresh_model,
    generate_batch_for_task,
    task_config_from_dict,
    tuned_response_metrics,
)

CONFIG_PATH = REPO / 'configs' / 'fixation_circular_working_memory.yaml'
OUTPUT_DIR = REPO / 'outputs' / 'fixation_circular_working_memory'
RUN_NAME = 'fixation_circular_working_memory'
GAIN_OUTPUT_DIR = OUTPUT_DIR / 'perturbation_experiments' / 'gain_modulation'
FIGURE_DIR = GAIN_OUTPUT_DIR / 'figures'
METRIC_DIR = GAIN_OUTPUT_DIR / 'metrics'

print(f'Repository: {REPO}')
print(f'Gain outputs: {GAIN_OUTPUT_DIR}')

## 2. Load the checkpoint ensemble and frozen evaluation batches

In [ ]:
config = load_config(CONFIG_PATH)
device_info = select_device(config['training'].get('device', 'auto'))
device = device_info.device

MODEL_SEEDS = (20260714, 20260715, 20260716, 20260717, 20260718)
CHECKPOINT_PATHS = {
    seed: (
        OUTPUT_DIR
        / 'seed_sweep'
        / f'seed_{seed}'
        / 'checkpoints'
        / f'{RUN_NAME}_seed_{seed}.pt'
    )
    for seed in MODEL_SEEDS
}
missing_checkpoints = [path for path in CHECKPOINT_PATHS.values() if not path.exists()]
if missing_checkpoints:
    raise FileNotFoundError(f'Missing checkpoints: {missing_checkpoints}')

models = {}
for seed, checkpoint_path in CHECKPOINT_PATHS.items():
    seed_config = deepcopy(config)
    seed_config['task']['seed'] = seed
    seed_model = fresh_model(seed_config, device)
    seed_checkpoint = torch.load(checkpoint_path, map_location=device)
    seed_model.load_state_dict(seed_checkpoint['model_state'])
    seed_model.eval()
    models[seed] = seed_model

N_EVAL_BATCHES = 4
EVAL_BATCH_SIZE = 64
EVAL_TASK_SEED = 202607230
evaluation_config = deepcopy(config)
evaluation_config['task']['seed'] = EVAL_TASK_SEED
evaluation_batches = []
for batch_index in range(N_EVAL_BATCHES):
    task_config = task_config_from_dict(
        evaluation_config,
        seed_offset=batch_index,
        batch_size=EVAL_BATCH_SIZE,
    )
    batch = generate_batch_for_task(task_config)
    inputs, targets, loss_mask = batch_to_tensors(batch, device)
    evaluation_batches.append((batch, inputs, targets, loss_mask))

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {device}')
print(f'Loaded checkpoints: {len(models)}')
print(f'Evaluation trials per checkpoint and condition: {N_EVAL_BATCHES * EVAL_BATCH_SIZE}')
print('All models are frozen and recurrent training noise is disabled.')

## 3. Global neural response gain

Global response gain multiplies the complete leaky update before `tanh`, changing the activation slope uniformly across units and throughout the trial:

$$h_t=\tanh\!\left(g_\phi\left[(1-\alpha)h_{t-1}+\alpha(W_xx_t+W_hh_{t-1})\right]\right).$$

In [ ]:
@torch.no_grad()
def forward_with_response_gain(model, inputs, response_gain: float):
    """Run the trained RNN with global response gain at every timestep."""
    if response_gain <= 0:
        raise ValueError('response_gain must be greater than zero.')
    hidden = model.rnn.init_hidden(inputs.shape, inputs.device)
    hidden_states = []
    for input_t in inputs:
        synaptic_drive = model.rnn.input2h(input_t) + model.rnn.h2h(hidden)
        leaky_update = (
            hidden * model.rnn.oneminusalpha
            + synaptic_drive * model.rnn.alpha
        )
        hidden = model.rnn._activation(response_gain * leaky_update)
        hidden_states.append(hidden)
    hidden_states = torch.stack(hidden_states)
    return model.readout(hidden_states), hidden_states


@torch.no_grad()
def evaluate_condition(model, forward_fn):
    """Aggregate behaviour and saturation over the frozen evaluation set."""
    angular_errors = []
    population_squared_errors = []
    fixation_accuracies = []
    saturated = 0
    total_states = 0
    for batch, inputs, targets, loss_mask in evaluation_batches:
        predictions, states = forward_fn(inputs)
        metrics = tuned_response_metrics(
            predictions,
            targets,
            loss_mask,
            batch.preferred_angles,
            batch.angles,
        )
        angular_errors.extend(metrics['angular_errors_degrees'])
        population_squared_errors.extend(metrics['population_squared_errors'])
        fixation_accuracies.append(metrics['fixation_accuracy'])
        saturated += int((states.abs() > 0.95).sum().item())
        total_states += states.numel()
    return {
        'mean_angular_error_degrees': float(np.mean(angular_errors)),
        'median_angular_error_degrees': float(np.median(angular_errors)),
        'population_mse': float(np.mean(population_squared_errors)),
        'fixation_accuracy': float(np.mean(fixation_accuracies)),
        'saturation_fraction': saturated / total_states,
    }


def joint_pca_projection(baseline_states, perturbed_states):
    """Project two trajectories into one jointly fitted two-dimensional PCA space."""
    baseline_np = baseline_states.detach().cpu().numpy()
    perturbed_np = perturbed_states.detach().cpu().numpy()
    combined = np.concatenate(
        [baseline_np.reshape(-1, baseline_np.shape[-1]), perturbed_np.reshape(-1, perturbed_np.shape[-1])],
        axis=0,
    )
    centre = combined.mean(axis=0, keepdims=True)
    _, singular_values, components = np.linalg.svd(combined - centre, full_matrices=False)
    axes = components[:2].T
    explained = singular_values[:2] ** 2 / np.sum(singular_values**2)
    baseline_projection = ((baseline_np.reshape(-1, baseline_np.shape[-1]) - centre) @ axes).reshape(
        baseline_np.shape[0], baseline_np.shape[1], 2
    )
    perturbed_projection = ((perturbed_np.reshape(-1, perturbed_np.shape[-1]) - centre) @ axes).reshape(
        perturbed_np.shape[0], perturbed_np.shape[1], 2
    )
    return baseline_projection, perturbed_projection, explained


def plot_gain_summary(results, x_column, x_label, baseline_states, perturbed_states, angles, title, perturbed_label, output_path):
    """Plot paired checkpoint performance and joint-space PCA trajectories."""
    checkpoint_means = (
        results.groupby(['seed', x_column], as_index=False)['delta_angular_error_degrees'].mean()
    )
    summary = checkpoint_means.groupby(x_column)['delta_angular_error_degrees'].agg(['mean', 'sem']).reset_index()
    baseline_projection, perturbed_projection, explained = joint_pca_projection(
        baseline_states, perturbed_states
    )

    figure, axes = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True)
    for seed, seed_rows in checkpoint_means.groupby('seed'):
        axes[0].plot(
            seed_rows[x_column],
            seed_rows['delta_angular_error_degrees'],
            color='0.68',
            marker='o',
            linewidth=1.0,
            alpha=0.8,
        )
    axes[0].errorbar(
        summary[x_column],
        summary['mean'],
        yerr=summary['sem'],
        color='tab:blue',
        marker='o',
        linewidth=2.4,
        capsize=3,
        label='Mean ± SEM (n=5 checkpoints)',
    )
    axes[0].axhline(0, color='0.15', linewidth=1, linestyle='--')
    axes[0].set_xlabel(x_label)
    axes[0].set_ylabel('Δ mean angular error from baseline (°)')
    axes[0].set_title('Performance effect')
    axes[0].legend(frameon=False, fontsize=8)
    axes[0].grid(alpha=0.2)

    trial_order = np.argsort(np.asarray(angles))
    n_trajectories = min(24, len(trial_order))
    trial_indices = trial_order[np.linspace(0, len(trial_order) - 1, n_trajectories, dtype=int)]
    normalizer = plt.Normalize(0, 2 * np.pi)
    for axis, projection, panel_title in zip(
        axes[1:],
        (baseline_projection, perturbed_projection),
        ('Baseline', perturbed_label),
    ):
        for trial_index in trial_indices:
            colour = plt.cm.hsv(normalizer(float(angles[trial_index]) % (2 * np.pi)))
            axis.plot(
                projection[:, trial_index, 0],
                projection[:, trial_index, 1],
                color=colour,
                linewidth=0.9,
                alpha=0.65,
            )
            axis.scatter(
                projection[-1, trial_index, 0],
                projection[-1, trial_index, 1],
                color=colour,
                s=12,
                edgecolors='none',
            )
        axis.set_xlabel(f'PC1 ({100 * explained[0]:.1f}%)')
        axis.set_ylabel(f'PC2 ({100 * explained[1]:.1f}%)')
        axis.set_title(panel_title)
        axis.grid(alpha=0.15)

    all_points = np.concatenate(
        [baseline_projection.reshape(-1, 2), perturbed_projection.reshape(-1, 2)], axis=0
    )
    lower = all_points.min(axis=0)
    upper = all_points.max(axis=0)
    padding = np.maximum((upper - lower) * 0.06, 1e-3)
    for axis in axes[1:]:
        axis.set_xlim(lower[0] - padding[0], upper[0] + padding[0])
        axis.set_ylim(lower[1] - padding[1], upper[1] + padding[1])

    colour_bar = figure.colorbar(
        plt.cm.ScalarMappable(norm=normalizer, cmap='hsv'),
        ax=axes[1:],
        fraction=0.035,
        pad=0.02,
    )
    colour_bar.set_label('Target angle (radians)')
    figure.suptitle(title)
    figure.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.show()
    return checkpoint_means, summary


baseline_by_seed = {
    seed: evaluate_condition(model, lambda inputs, model=model: model(inputs))
    for seed, model in models.items()
}
baseline_errors = np.array(
    [baseline_by_seed[seed]['mean_angular_error_degrees'] for seed in MODEL_SEEDS]
)
median_baseline = float(np.median(baseline_errors))
REFERENCE_SEED = min(
    MODEL_SEEDS,
    key=lambda seed: abs(baseline_by_seed[seed]['mean_angular_error_degrees'] - median_baseline),
)

RESPONSE_GAIN_VALUES = (0.80, 0.90, 1.00, 1.10, 1.20)
response_rows = []
for seed, model in models.items():
    baseline = baseline_by_seed[seed]
    baseline_predictions, baseline_states = model(evaluation_batches[0][1])
    neutral_predictions, neutral_states = forward_with_response_gain(
        model, evaluation_batches[0][1], response_gain=1.0
    )
    torch.testing.assert_close(neutral_predictions, baseline_predictions)
    torch.testing.assert_close(neutral_states, baseline_states)
    for gain in RESPONSE_GAIN_VALUES:
        condition = evaluate_condition(
            model,
            lambda inputs, model=model, gain=gain: forward_with_response_gain(model, inputs, gain),
        )
        response_rows.append(
            {
                'seed': seed,
                'response_gain': gain,
                **condition,
                'baseline_mean_angular_error_degrees': baseline['mean_angular_error_degrees'],
                'delta_angular_error_degrees': (
                    condition['mean_angular_error_degrees'] - baseline['mean_angular_error_degrees']
                ),
            }
        )

response_results = pd.DataFrame(response_rows)
response_results.to_csv(METRIC_DIR / 'global_response_gain_sweep.csv', index=False)
reference_model = models[REFERENCE_SEED]
pca_batch, pca_inputs, _, _ = evaluation_batches[0]
with torch.no_grad():
    _, response_baseline_states = reference_model(pca_inputs)
response_pca_gain = 1.20
_, response_perturbed_states = forward_with_response_gain(
    reference_model, pca_inputs, response_gain=response_pca_gain
)
response_checkpoint_means, response_summary = plot_gain_summary(
    response_results,
    x_column='response_gain',
    x_label='Global response gain',
    baseline_states=response_baseline_states,
    perturbed_states=response_perturbed_states,
    angles=pca_batch.angles,
    title=f'Global response gain across five checkpoints; PCA seed {REFERENCE_SEED}',
    perturbed_label=f'Response gain = {response_pca_gain:.2f}',
    output_path=FIGURE_DIR / 'global_response_gain.png',
)
print(response_summary.to_string(index=False))

## 4. Fixed heterogeneous neural response gain

Each unit receives one deterministic positive gain that remains fixed across the entire trial and evaluation set. Population mean gain is held exactly at one, so the sweep isolates unit-to-unit heterogeneity. Three gain-vector seeds are averaged within each checkpoint before checkpoint-level inference.

In [ ]:
def make_fixed_response_gains(hidden_size, mean_gain, log_std, seed, device):
    """Return a deterministic positive gain vector with an exact mean."""
    if mean_gain <= 0 or log_std < 0:
        raise ValueError('mean_gain must be positive and log_std non-negative.')
    generator = torch.Generator(device='cpu').manual_seed(seed)
    standard_normal = torch.randn(hidden_size, generator=generator)
    gains = torch.exp(log_std * standard_normal - 0.5 * log_std**2)
    gains = gains * (mean_gain / gains.mean())
    return gains.to(device=device)


@torch.no_grad()
def forward_with_heterogeneous_response_gain(model, inputs, response_gains):
    """Run the RNN with one fixed response gain per hidden unit."""
    response_gains = torch.as_tensor(response_gains, device=inputs.device, dtype=inputs.dtype)
    if response_gains.shape != (model.rnn.hidden_size,):
        raise ValueError('response_gains must have shape (hidden_size,).')
    if torch.any(response_gains <= 0):
        raise ValueError('All response gains must be greater than zero.')
    hidden = model.rnn.init_hidden(inputs.shape, inputs.device)
    hidden_states = []
    for input_t in inputs:
        synaptic_drive = model.rnn.input2h(input_t) + model.rnn.h2h(hidden)
        leaky_update = (
            hidden * model.rnn.oneminusalpha
            + synaptic_drive * model.rnn.alpha
        )
        hidden = model.rnn._activation(response_gains * leaky_update)
        hidden_states.append(hidden)
    hidden_states = torch.stack(hidden_states)
    return model.readout(hidden_states), hidden_states


HETEROGENEOUS_LOG_STDS = (0.000, 0.025, 0.050, 0.100, 0.150)
HETEROGENEOUS_VECTOR_SEEDS = (3101, 3102, 3103)
HETEROGENEOUS_MEAN_GAIN = 1.0
heterogeneous_rows = []
for seed, model in models.items():
    baseline = baseline_by_seed[seed]
    baseline_predictions, baseline_states = model(evaluation_batches[0][1])
    ones = torch.ones(model.rnn.hidden_size, device=device)
    neutral_predictions, neutral_states = forward_with_heterogeneous_response_gain(
        model, evaluation_batches[0][1], ones
    )
    torch.testing.assert_close(neutral_predictions, baseline_predictions)
    torch.testing.assert_close(neutral_states, baseline_states)
    for log_std in HETEROGENEOUS_LOG_STDS:
        for gain_vector_seed in HETEROGENEOUS_VECTOR_SEEDS:
            gains = make_fixed_response_gains(
                model.rnn.hidden_size,
                HETEROGENEOUS_MEAN_GAIN,
                log_std,
                gain_vector_seed,
                device,
            )
            condition = evaluate_condition(
                model,
                lambda inputs, model=model, gains=gains: (
                    forward_with_heterogeneous_response_gain(model, inputs, gains)
                ),
            )
            heterogeneous_rows.append(
                {
                    'seed': seed,
                    'gain_vector_seed': gain_vector_seed,
                    'heterogeneous_log_std': log_std,
                    'gain_mean': float(gains.mean().item()),
                    'gain_std': float(gains.std().item()),
                    'gain_min': float(gains.min().item()),
                    'gain_max': float(gains.max().item()),
                    **condition,
                    'baseline_mean_angular_error_degrees': baseline['mean_angular_error_degrees'],
                    'delta_angular_error_degrees': (
                        condition['mean_angular_error_degrees'] - baseline['mean_angular_error_degrees']
                    ),
                }
            )

heterogeneous_results = pd.DataFrame(heterogeneous_rows)
heterogeneous_results.to_csv(METRIC_DIR / 'heterogeneous_response_gain_sweep.csv', index=False)
heterogeneous_pca_log_std = 0.10
heterogeneous_pca_vector_seed = HETEROGENEOUS_VECTOR_SEEDS[0]
reference_gains = make_fixed_response_gains(
    reference_model.rnn.hidden_size,
    HETEROGENEOUS_MEAN_GAIN,
    heterogeneous_pca_log_std,
    heterogeneous_pca_vector_seed,
    device,
)
with torch.no_grad():
    _, heterogeneous_baseline_states = reference_model(pca_inputs)
_, heterogeneous_perturbed_states = forward_with_heterogeneous_response_gain(
    reference_model, pca_inputs, reference_gains
)
heterogeneous_checkpoint_means, heterogeneous_summary = plot_gain_summary(
    heterogeneous_results,
    x_column='heterogeneous_log_std',
    x_label='Gain heterogeneity (log-SD)',
    baseline_states=heterogeneous_baseline_states,
    perturbed_states=heterogeneous_perturbed_states,
    angles=pca_batch.angles,
    title=f'Heterogeneous response gain across five checkpoints; PCA seed {REFERENCE_SEED}',
    perturbed_label=f'Log-SD = {heterogeneous_pca_log_std:.2f}',
    output_path=FIGURE_DIR / 'heterogeneous_response_gain.png',
)
print(heterogeneous_summary.to_string(index=False))

## 5. Recurrent-drive gain

Recurrent gain scales only the learned weight-dependent feedback $W_hh_{t-1}$. Input drive, the carried leaky state, activation function, and learned recurrent bias remain unchanged:

$$h_t=\phi\!\left((1-\alpha)h_{t-1}+\alpha[W_xx_t+b_x+g_rW_hh_{t-1}+b_h]\right).$$

In [ ]:
@torch.no_grad()
def forward_with_recurrent_gain(model, inputs, recurrent_gain: float):
    """Run the RNN while scaling only the recurrent weight contribution."""
    if recurrent_gain <= 0:
        raise ValueError('recurrent_gain must be greater than zero.')
    hidden = model.rnn.init_hidden(inputs.shape, inputs.device)
    hidden_states = []
    for input_t in inputs:
        input_affine = model.rnn.input2h(input_t)
        recurrent_weight_drive = torch.nn.functional.linear(
            hidden, model.rnn.h2h.weight, bias=None
        )
        synaptic_drive = input_affine + recurrent_gain * recurrent_weight_drive
        if model.rnn.h2h.bias is not None:
            synaptic_drive = synaptic_drive + model.rnn.h2h.bias
        leaky_update = (
            hidden * model.rnn.oneminusalpha
            + synaptic_drive * model.rnn.alpha
        )
        hidden = model.rnn._activation(leaky_update)
        hidden_states.append(hidden)
    hidden_states = torch.stack(hidden_states)
    return model.readout(hidden_states), hidden_states


RECURRENT_GAIN_VALUES = (0.80, 0.90, 1.00, 1.10, 1.20)
recurrent_rows = []
for seed, model in models.items():
    baseline = baseline_by_seed[seed]
    baseline_predictions, baseline_states = model(evaluation_batches[0][1])
    neutral_predictions, neutral_states = forward_with_recurrent_gain(
        model, evaluation_batches[0][1], recurrent_gain=1.0
    )
    torch.testing.assert_close(neutral_predictions, baseline_predictions)
    torch.testing.assert_close(neutral_states, baseline_states)
    for gain in RECURRENT_GAIN_VALUES:
        condition = evaluate_condition(
            model,
            lambda inputs, model=model, gain=gain: forward_with_recurrent_gain(model, inputs, gain),
        )
        recurrent_rows.append(
            {
                'seed': seed,
                'recurrent_gain': gain,
                **condition,
                'baseline_mean_angular_error_degrees': baseline['mean_angular_error_degrees'],
                'delta_angular_error_degrees': (
                    condition['mean_angular_error_degrees'] - baseline['mean_angular_error_degrees']
                ),
            }
        )

recurrent_results = pd.DataFrame(recurrent_rows)
recurrent_results.to_csv(METRIC_DIR / 'recurrent_gain_sweep.csv', index=False)
recurrent_pca_gain = 1.20
with torch.no_grad():
    _, recurrent_baseline_states = reference_model(pca_inputs)
_, recurrent_perturbed_states = forward_with_recurrent_gain(
    reference_model, pca_inputs, recurrent_gain=recurrent_pca_gain
)
recurrent_checkpoint_means, recurrent_summary = plot_gain_summary(
    recurrent_results,
    x_column='recurrent_gain',
    x_label='Recurrent-drive gain',
    baseline_states=recurrent_baseline_states,
    perturbed_states=recurrent_perturbed_states,
    angles=pca_batch.angles,
    title=f'Recurrent-drive gain across five checkpoints; PCA seed {REFERENCE_SEED}',
    perturbed_label=f'Recurrent gain = {recurrent_pca_gain:.2f}',
    output_path=FIGURE_DIR / 'recurrent_gain.png',
)
print(recurrent_summary.to_string(index=False))